In [ ]:

import numpy as np
import os
from scipy import stats
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load the preprocessed data and age and sex adjusted data
os.chdir("D:\Raman analysis files\Substudy II analysis\Manuscript analysis files")

data = np.load('spectral_data_gender_and_age_adjusted.npy')
class_labels  = np.load('class_labels_averaged.npy')
sample_ids = np.load('sample_ids.npy')
# Dowload wavenumbers for plotting
truncated_wavenumbers_control = np.load('wavenumbers_control.npy', allow_pickle=True)
truncated_wavenumbers_RA = np.load('wavenumbers_RA.npy', allow_pickle=True)

In [ ]:
# Download clinical data excel file
clinical_data = pd.read_csv('cleaned_clinical_data.csv')

In [ ]:
# Select seropositive sample ids from clinical data as numpy array
seropositive_sample_ids = clinical_data[clinical_data['ICD10'].str.startswith('M05')]['sample_ID'].values.astype(str)
seronegative_sample_ids = clinical_data[clinical_data['ICD10'].str.startswith('M06')]['sample_ID'].values.astype(str)

# Select female control sample ids from clinical data as numpy array
female_control_sample_ids = clinical_data[(clinical_data['GENDER'] == 'Female') & (clinical_data['ICD10'] == 'CONTROL')]['sample_ID'].values.astype(str)
# Select male control sample ids from clinical data as numpy array
male_control_sample_ids = clinical_data[(clinical_data['GENDER'] == 'Male') & (clinical_data['ICD10'] == 'CONTROL')]['sample_ID'].values.astype(str)
# Select RA female sample ids from clinical data as numpy array
female_RA_sample_ids = clinical_data[(clinical_data['GENDER'] == 'Female') & (clinical_data['ICD10'].str.startswith('M05') | clinical_data['ICD10'].str.startswith('M06'))]['sample_ID'].values.astype(str)
# Select RA male sample ids from clinical data as numpy array
male_RA_sample_ids = clinical_data[(clinical_data['GENDER'] == 'Male') & (clinical_data['ICD10'].str.startswith('M05') | clinical_data['ICD10'].str.startswith('M06'))]['sample_ID'].values.astype(str)

# select the indices of the seropositive and seronegative samples in the sample_ids vector
seropositive_ind = np.where(np.isin(sample_ids, seropositive_sample_ids))[0]
seronegative_ind = np.where(np.isin(sample_ids, seronegative_sample_ids))[0]
female_control_ind = np.where(np.isin(sample_ids, female_control_sample_ids))[0]
male_control_ind = np.where(np.isin(sample_ids, male_control_sample_ids))[0]
female_RA_ind = np.where(np.isin(sample_ids, female_RA_sample_ids))[0]
male_RA_ind = np.where(np.isin(sample_ids, male_RA_sample_ids))[0]

data_females_control = data[female_control_ind, :]
data_males_control = data[male_control_ind, :]
data_females_RA = data[female_RA_ind, :]
data_males_RA = data[male_RA_ind, :]

labels_females_control = class_labels[female_control_ind]
labels_males_control = class_labels[male_control_ind]
labels_females_RA = class_labels[female_RA_ind]
labels_males_RA = class_labels[male_RA_ind]

data_RA_pos = data[seropositive_ind, :]
data_RA_neg = data[seronegative_ind, :]


In [ ]:
#Perform Bonferroni and Benjamini/Hochberg correctecd t-test between control and RA groupn to find statistically significant wavenumbers. Change the data variable according the groups being compared.
control_data = data[class_labels == 0]
ra_data = data[class_labels == 1]
control_data = np.asarray(control_data, dtype=float)
ra_data = np.asarray(ra_data, dtype=float)

p_value_to_test = 0.05
# Mann-Whitney U test
p_values = []
for i in range(1245):
    stat, p = stats.mannwhitneyu(control_data[:, i], ra_data[:, i])
    p_values.append(p)

# False Discovery Rate (FDR) correction using Benjamini-Hochberg procedure with scipy
from statsmodels.stats.multitest import multipletests
reject_bh, pvals_corrected_bh, _, _ = multipletests(p_values, alpha=p_value_to_test, method='fdr_bh')   
significant_wavenumbers = np.where(reject_bh)[0]
print("Significant wavenumbers after Benjamini-Hochberg correction:", significant_wavenumbers)

# False Discovery Rate (FDR) correction using Bonferroni procedure with scipy
from statsmodels.stats.multitest import multipletests
reject_bonferroni, pvals_corrected_bonferroni, _, _ = multipletests(p_values, alpha=p_value_to_test, method='bonferroni')   
significant_wavenumbers = np.where(reject_bonferroni)[0]
print("Significant wavenumbers after Bonferroni correction:", significant_wavenumbers)

